In [1]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

/usr/local/lib/python3.12/site-packages/leidenalg/VertexPartition.py:388: SyntaxWarning: invalid escape sequence '\m'
  """ Implements modularity. This quality function is well-defined only for positive edge weights.
/usr/local/lib/python3.12/site-packages/leidenalg/VertexPartition.py:761: SyntaxWarning: invalid escape sequence '\m'
  """ Implements Reichardt and Bornholdt's Potts model with a configuration null model.
/usr/local/lib/python3.12/site-packages/leidenalg/Optimiser.py:7: SyntaxWarning: invalid escape sequence '\g'
  """ Class for doing community detection using the Leiden algorithm.
/usr/local/lib/python3.12/site-packages/leidenalg/Optimiser.py:305: SyntaxWarning: invalid escape sequence '\s'
  """ Optimise the given partitions simultaneously.


In [ ]:
adata_path = '20250811_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.annot_corrected.index_dedup.cg.h5ad'  # CC RNA anndata, change name and path as necessarry
adata = sc.read(adata_path)
adata

Only considering the two last: ['.cg', '.h5ad'].
Only considering the two last: ['.cg', '.h5ad'].


AnnData object with n_obs × n_vars = 751541 × 39514
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'n_counts', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'region', 'vireo_assignment', 'library', 'subclass_scANVI', 'cell_type_scANVI', 'biobank_name', 'sex', 'age_at_collection', 'primary_diagnosis', 'age_at_diagnosis', 'batch', 'new_index', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'doublet_score', 'predicted_doublet', 'experiment', 'prob_max', 'prob_doublet', 'n_vars', 'doublet_logLikRatio', 'leiden_2.5', 'cell_type_ma

In [3]:
sc.tl.rank_genes_groups(adata, groupby = 'cell_type_broad', method = 'wilcoxon', layer = 'log1p_norm')

In [4]:
sc.get.rank_genes_groups_df(adata, group="Oligo").head(20)


,names,scores,logfoldchanges,pvals,pvals_adj
0,ST18,517.639343,6.345450,0.0,0.0
1,CTNNA3,489.440887,4.238723,0.0,0.0
2,RNF220,483.746582,4.547774,0.0,0.0
3,MBP,464.298981,3.773790,0.0,0.0
4,PIP4K2A,454.928345,3.388988,0.0,0.0
5,TMEM144,443.175354,4.918151,0.0,0.0
6,C10orf90,417.995789,4.079338,0.0,0.0
7,PLP1,411.608307,5.508069,0.0,0.0
8,DOCK5,408.179962,3.538615,0.0,0.0
9,FRMD4B,404.398590,3.742778,0.0,0.0


In [5]:
ranked_genes = sc.get.rank_genes_groups_df(adata, group="Oligo")
ranked_genes['abs_logfoldchanges'] = ranked_genes['logfoldchanges'].abs()


In [8]:
ranked_genes

,names,scores,logfoldchanges,pvals,pvals_adj,abs_logfoldchanges
0,ST18,517.639343,6.345450,0.0,0.0,6.345450
1,CTNNA3,489.440887,4.238723,0.0,0.0,4.238723
2,RNF220,483.746582,4.547774,0.0,0.0,4.547774
3,MBP,464.298981,3.773790,0.0,0.0,3.773790
4,PIP4K2A,454.928345,3.388988,0.0,0.0,3.388988
...,...,...,...,...,...,...
39509,NALF1,-451.473602,-5.866171,0.0,0.0,5.866171
39510,CELF2,-453.642731,-5.770234,0.0,0.0,5.770234
39511,KCNMA1,-456.038361,-5.763328,0.0,0.0,5.763328
39512,NRXN1,-469.242157,-5.824150,0.0,0.0,5.824150


In [9]:
adata.obs['cell_type_broad'].value_counts()

cell_type_broad
ExN          260309
InN          137135
Astro        122948
Oligo        116404
OPC           59762
Micro-PVM     51148
Endo           3835
Name: count, dtype: int64

In [10]:
cluster_key = 'cell_type_broad'
cluster_of_interest = 'Oligo'#'Oligo'
adata.X = adata.layers['log1p_norm']

adata_cluster = adata[adata.obs[cluster_key] == cluster_of_interest]
mean_expression = adata_cluster.X.mean(axis=0)

if hasattr(mean_expression, "A1"):
    mean_expression = mean_expression.A1  # for sparse matrices
elif isinstance(mean_expression, np.matrix):
    mean_expression = np.asarray(mean_expression).flatten()
else:
    mean_expression = np.ravel(mean_expression)
    
gene_expression = pd.DataFrame({
    'gene': adata.var_names.to_list(),
    'mean_expression': mean_expression
})

In [11]:
gene_expression_sorted = gene_expression.sort_values(by='mean_expression', ascending=False)
gene_expression_sorted

,gene,mean_expression
26980,MALAT1,4.028512
30567,PCDH9,3.146179
9272,IL1RAPL1,2.688106
32337,QKI,2.216876
5063,DLG2,2.167451
...,...,...
37137,TRAV6,0.000000
37138,TRAV7,0.000000
37139,TRAV8-1,0.000000
37140,TRAV8-2,0.000000


In [12]:
gene_expression_sorted[:30]

,gene,mean_expression
26980,MALAT1,4.028512
30567,PCDH9,3.146179
9272,IL1RAPL1,2.688106
32337,QKI,2.216876
5063,DLG2,2.167451
4386,CTNNA3,2.144596
30721,PDE4B,2.104969
26962,MAGI2,2.104294
35414,ST18,2.086291
2462,CADM2,2.062403


In [13]:
n_genes = len(gene_expression_sorted)
top_n = int(n_genes * 0.1)
top_10_percent = gene_expression_sorted.head(top_n)
bottom_10_percent = gene_expression_sorted.tail(top_n)

In [ ]:
# make sure output dir exists
os.makedirs('expression_info', exist_ok=True)
top_10_percent['gene'].to_csv(f'expression_info/cc_{cluster_of_interest}_top10_expressed_genes.csv', index = False)

In [15]:
bottom_10_percent['gene'].to_csv(f'expression_info/cc_{cluster_of_interest}_bottom10_expressed_genes.csv', index = False)

In [16]:
pip list

Package                   Version
------------------------- --------------
absl-py                   2.1.0
adjustText                1.3.0
aiobotocore               2.17.0
aiohappyeyeballs          2.4.4
aiohttp                   3.11.11
aiohttp-retry             2.9.1
aioitertools              0.12.0
aiosignal                 1.3.2
amqp                      5.3.1
anndata                   0.11.2
anndata2ri                1.3.2
annotated-types           0.7.0
ansicolors                1.1.8
antlr4-python3-runtime    4.9.3
anyio                     4.8.0
appdirs                   1.4.4
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
array_api_compat          1.10.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.4
asyncssh                  2.19.0
atpublic                  5.0
attrs                     24.3.0
babel                     2.16.0
beautifulsoup4            4.12.3
billiard                  4.2.1
biopython          